# Foraging Cache — DuckDB Query Examples

The parquet database lives at:
- `scratch/foraging_cache/session_table_sample.parquet` — flat session table
- `scratch/foraging_cache/trial_table/` — Hive-partitioned by `subject_id`
- `scratch/foraging_cache/event_table/` — Hive-partitioned by `subject_id`

**Primary pattern**: filter the session table, then JOIN to trial/event tables so every
row carries the session-level metadata (subject, date, task, foraging_eff, …) alongside the trial data.

Three options are needed when reading the partitioned tables:
- `hive_partitioning=true` — enables partition-level pruning on `subject_id`
- `union_by_name=true` — merges column schemas across files; different NWB readers produce different column sets, missing columns fill with NULL
- `CAST(subject_id AS VARCHAR)` — DuckDB infers the `subject_id=<n>` partition directory name as BIGINT; cast to VARCHAR to compare against the session table

In [1]:
import duckdb
import pandas as pd

SCRATCH      = "/root/capsule/scratch/foraging_cache"
SESSION_OUT  = f"{SCRATCH}/session_table_sample.parquet"
TRIAL_OUT    = f"{SCRATCH}/trial_table"
EVENT_OUT    = f"{SCRATCH}/event_table"

# Reusable snippets — union_by_name handles schema differences across NWB reader paths
READ_TRIALS = f"read_parquet('{TRIAL_OUT}/**/*.parquet', hive_partitioning=true, union_by_name=true)"
READ_EVENTS = f"read_parquet('{EVENT_OUT}/**/*.parquet', hive_partitioning=true, union_by_name=true)"

## 1. Browse the session table

In [2]:
duckdb.sql(f"""
    SELECT _session_id, subject_id, session_date, task, finished_trials, foraging_eff
    FROM read_parquet('{SESSION_OUT}')
    ORDER BY session_date DESC
    LIMIT 10
""").df()

,_session_id,subject_id,session_date,task,finished_trials,foraging_eff
0,810888_2025-09-26_85905,810888,2025-09-26,Uncoupled Baiting,369,0.681390
1,811021_2025-09-25_91453,811021,2025-09-25,Uncoupled Baiting,527,0.674880
2,812307_2025-09-24_90657,812307,2025-09-24,Uncoupled Baiting,522,0.698302
3,801997_2025-09-23_94102,801997,2025-09-23,Uncoupled Baiting,533,0.782237
4,810756_2025-09-22_130932,810756,2025-09-22,Uncoupled Baiting,494,0.656980
5,806158_2025-09-17_135453,806158,2025-09-17,Uncoupled Without Baiting,520,0.671698
6,801997_2025-09-09_92324,801997,2025-09-09,Uncoupled Baiting,527,0.696414
7,804764_2025-09-08_104034,804764,2025-09-08,Uncoupled Baiting,538,0.630717
8,806664_2025-09-05_191422,806664,2025-09-05,Uncoupled Baiting,542,0.717902
9,808057_2025-09-03_124512,808057,2025-09-03,Uncoupled Baiting,313,0.812956


## 2. Primary use case — filter sessions → trials with session keys merged

Filter the session table however you like, then JOIN to the trial table so every trial
row already carries the session-level metadata.

In [8]:
df_trials = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_OUT}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        t.session_id,
        t.trial,
        t.animal_response,
        t.earned_reward,
        t.reward_probabilityL,
        t.reward_probabilityR,
        t.rewarded_historyL,
        t.rewarded_historyR
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date
""").df()

print(f"{len(df_trials)} trials from {df_trials['session_id'].nunique()} sessions")
df_trials.head(10)

7458 trials from 17 sessions


,subject_id,session_date,session_id,trial,animal_response,earned_reward,reward_probabilityL,reward_probabilityR,rewarded_historyL,rewarded_historyR
0,758018,2025-03-12,758018_2025-03-12_143816,0,1.0,False,0.1,0.5,False,False
1,758018,2025-03-12,758018_2025-03-12_143816,1,1.0,True,0.1,0.5,False,True
2,758018,2025-03-12,758018_2025-03-12_143816,2,1.0,False,0.1,0.5,False,False
3,758018,2025-03-12,758018_2025-03-12_143816,3,0.0,False,0.1,0.5,False,False
4,758018,2025-03-12,758018_2025-03-12_143816,4,0.0,False,0.1,0.5,False,False
5,758018,2025-03-12,758018_2025-03-12_143816,5,0.0,False,0.1,0.5,False,False
6,758018,2025-03-12,758018_2025-03-12_143816,6,0.0,True,0.1,0.5,True,False
7,758018,2025-03-12,758018_2025-03-12_143816,7,0.0,False,0.1,0.5,False,False
8,758018,2025-03-12,758018_2025-03-12_143816,8,0.0,True,0.1,0.5,True,False
9,758018,2025-03-12,758018_2025-03-12_143816,9,0.0,False,0.1,0.5,False,False


## 3. Same pattern for events

In [9]:
df_events = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_OUT}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        s.task,
        e.session_id,
        e.timestamps,
        e.event,
        e.data
    FROM {READ_EVENTS} e
    JOIN sel s ON e.session_id = s._session_id
    WHERE CAST(e.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date, e.timestamps
""").df()

print(f"{len(df_events)} events from {df_events['session_id'].nunique()} sessions")
print(f"Event types: {sorted(df_events['event'].unique())}")
df_events.head(10)

80937 events from 17 sessions
Event types: ['goCue_start_time', 'left_lick_time', 'left_reward_delivery_time', 'optogenetics_time', 'right_lick_time', 'right_reward_delivery_time']


,subject_id,session_date,task,session_id,timestamps,event,data
0,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,-4.777280,right_lick_time,1.0
1,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,0.000000,goCue_start_time,1
2,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,0.160512,right_lick_time,1.0
3,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,0.332864,right_lick_time,1.0
4,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,0.515712,right_lick_time,1.0
5,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,0.704608,right_lick_time,1.0
6,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,0.879872,right_lick_time,1.0
7,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,1.036032,right_lick_time,1.0
8,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,1.231712,left_lick_time,1.0
9,758018,2025-03-12,Uncoupled Without Baiting,758018_2025-03-12_143816,1.385472,left_lick_time,1.0


## 4. Aggregate across sessions — all in SQL

In [10]:
duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_OUT}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        COUNT(DISTINCT s._session_id)       AS n_sessions,
        COUNT(*)                            AS n_trials,
        AVG(t.earned_reward::DOUBLE)        AS mean_reward_rate,
        AVG(s.foraging_eff)                 AS mean_foraging_eff
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    GROUP BY s.subject_id
    ORDER BY mean_foraging_eff DESC
""").df()

,subject_id,n_sessions,n_trials,mean_reward_rate,mean_foraging_eff
0,791094,1,473,0.427061,0.896582
1,784806,1,548,0.510949,0.868756
2,775510,1,335,0.543284,0.856709
3,792292,1,415,0.544578,0.842159
4,763590,1,402,0.587065,0.841055
5,778103,1,318,0.569182,0.838850
6,758435,2,921,0.541802,0.833452
7,781895,1,391,0.496164,0.824482
8,787622,1,381,0.393701,0.813895
9,765628,1,394,0.449239,0.813863
